<div style="
    background-color:#2c3e50; 
    color:#ecf0f1; 
    font-weight:bold; 
    padding:20px 30px; 
    font-size:20px; 
    border-radius:8px; 
    text-align:center;
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    letter-spacing: 0.5px;
">
    Audio record frequency analysis
</div>

<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Extracts list of top dominant frequencies: <br>
    Path do audio to analyse needs to be provided <br>
    Need to provide start and end time segment from which the dominant frequencies will be extracted. Segment time was decided after manual evaluation using <span style="color: red; font-weight: bold;">"listen_audio_file.py"</span>
</h3>

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from IPython.display import Audio, display

# ----------------- Configuration -----------------
# Provide the path to your audio file here
file_path = "./audio/washing_machine/becken_BWM5381IX/others/wm_becken_BWM5381IX_warm_delicate_30_0.wav"

# Define time segment to analyze (in seconds)
# You can modify these values to analyze different segments
start_time = 2703  # Start time in seconds
end_time =  2708  # Start time in seconds

# -------------------------------------------------

y, sr = librosa.load(file_path, sr=11025)  # Load at 11025Hz

print(f"Audio loaded: {len(y)/sr:.2f} seconds total duration")
print(f"Sample rate: {sr} Hz")


segment_duration = end_time - start_time

print(f"\nAnalyzing segment from {start_time:.2f}s to {end_time:.2f}s ({segment_duration:.2f}s duration)")

# Convert time to sample indices
start_sample = int(start_time * sr)
end_sample = int(end_time * sr)

# Check if the requested time segment is within the audio file
if start_sample >= len(y):
    print(f"Error: Start time {start_time:.2f}s is beyond audio duration ({len(y)/sr:.2f}s)")
    exit()

if end_sample > len(y):
    print(f"Warning: End time {end_time:.2f}s exceeds audio duration, trimming to {len(y)/sr:.2f}s")
    end_sample = len(y)
    end_time = len(y) / sr

# Extract the specified time segment
y_segment = y[start_sample:end_sample]
print(f"Extracted {len(y_segment)/sr:.2f} seconds of audio")

# Analysis parameters
frame_size = 8192   # Matches FFT size
hop_size = 2048     # 75% overlap (good balance)

# Step 1: Energy envelope of the segment
env = np.array([
    np.sum(np.abs(y_segment[i:i+frame_size])**2)
    for i in range(0, len(y_segment) - frame_size, hop_size)
])

if len(env) == 0:
    print("Error: Segment too short for analysis")
    exit()

env /= np.max(env)

# Step 2: Autocorrelation
auto = np.correlate(env, env, mode='full')[len(env):]
auto /= np.max(auto)

# Step 3: Estimate dominant repeating interval
peaks, _ = find_peaks(auto, height=0.3, distance=sr*1/hop_size)
if len(peaks) == 0:
    print("No repeating patterns found in this segment.")
    # Analyze the entire segment if no patterns found
    analysis_segment = y_segment
    print("Analyzing entire selected segment for frequency content...")
else:
    lag_frames = peaks[0]
    lag_seconds = lag_frames * hop_size / sr
    print(f"Estimated repeating cycle duration: {lag_seconds:.2f} seconds")

    # Get the first complete cycle
    cycle_samples = int(lag_seconds * sr)
    if cycle_samples * 2 > len(y_segment):
        print("Warning: Cycle too long for segment, using entire segment")
        analysis_segment = y_segment
    else:
        analysis_segment = y_segment[cycle_samples:2*cycle_samples]
        print(f"Analyzing first cycle ({lag_seconds:.2f}s duration)")

# Analyze frequency content
# Compute FFT
n_fft = 16384  # Higher resolution for frequency analysis
fft = np.fft.rfft(analysis_segment, n=n_fft)
magnitudes = np.abs(fft)

# Convert to dB scale
magnitudes_db = 20 * np.log10(magnitudes + 1e-6)  # Add small value to avoid log(0)

# Get frequency bins
freqs = np.fft.rfftfreq(n_fft, d=1/sr)

# Find peaks (most prominent frequencies)
peaks, _ = find_peaks(magnitudes_db, height=np.mean(magnitudes_db))

# Get top 5 frequencies
top_n = 200
if len(peaks) > 0:
    top_idx = np.argsort(magnitudes_db[peaks])[-top_n:][::-1]
    top_freqs = freqs[peaks][top_idx]
    top_mags = magnitudes_db[peaks][top_idx]
else:
    # Fallback if no peaks found - just take highest magnitudes
    top_idx = np.argsort(magnitudes_db)[-top_n:][::-1]
    top_freqs = freqs[top_idx]
    top_mags = magnitudes_db[top_idx]

# Print results
print(f"\nTop {top_n} dominant frequencies in selected segment:")
for i, (freq, mag) in enumerate(zip(top_freqs, top_mags), 1):
    print(f"{i}. {freq:.1f} Hz ({mag:.1f} dB)")

# Plot frequency spectrum and spectrogram
plt.figure(figsize=(15, 12))

# Subplot 1: Waveform of selected segment
plt.subplot(3, 1, 1)
time_axis = np.linspace(start_time, start_time + len(y_segment)/sr, len(y_segment))
plt.plot(time_axis, y_segment)
plt.title(f'Waveform: {start_time:.2f}s to {end_time:.2f}s')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.grid(True)

# Subplot 2: Frequency spectrum
plt.subplot(3, 1, 2)
plt.plot(freqs, magnitudes_db, label='Frequency Spectrum', alpha=0.7)
plt.scatter(top_freqs, top_mags, color='red', s=50, zorder=5, label='Dominant Frequencies')
for i, (freq, mag) in enumerate(zip(top_freqs, top_mags), 1):
    plt.annotate(f'{i}: {freq:.1f} Hz', 
                xy=(freq, mag), 
                xytext=(10, 10), 
                textcoords='offset points',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.title(f'Frequency Analysis: {start_time:.2f}s to {end_time:.2f}s')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.xlim(0, min(2000, sr//2))  # Limit to 2kHz for better visibility
plt.grid(True)
plt.legend()

# Subplot 3: Spectrogram
plt.subplot(3, 1, 3)
# Compute spectrogram using librosa
hop_length = 4096
n_fft = 8192
D = librosa.stft(y_segment, n_fft=n_fft, hop_length=hop_length)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

# Create time and frequency axes for spectrogram
times = librosa.frames_to_time(np.arange(S_db.shape[1]), sr=sr, hop_length=hop_length) + start_time
freqs_spec = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

# Plot spectrogram
img = plt.imshow(S_db, aspect='auto', origin='lower', 
                extent=[times[0], times[-1], freqs_spec[0], freqs_spec[-1]],
                cmap='viridis')
plt.colorbar(img, format='%+2.0f dB')
plt.title(f'Spectrogram: {start_time:.2f}s to {end_time:.2f}s')
plt.xlabel('Time (seconds)')
plt.ylabel('Frequency (Hz)')
plt.ylim(0, min(500, sr//2))  # Limit to 2kHz for better visibility

plt.tight_layout()
plt.show()

# Play the analyzed segment
print(f"\nPlaying selected segment ({start_time:.2f}s to {end_time:.2f}s):")
display(Audio(y_segment, rate=sr))

# Summary statistics
print(f"\n=== Analysis Summary ===")
print(f"Sample rate: {sr} Hz")
print(f"Time segment: {start_time:.2f}s - {end_time:.2f}s ({segment_duration:.2f}s)")
print(f"Sample range: {start_sample} - {end_sample}")
print(f"Dominant frequency: {top_freqs[0]:.1f} Hz")
print(f"Energy in segment: {np.sum(y_segment**2):.2e}")
print(f"Peak amplitude: {np.max(np.abs(y_segment)):.3f}")

<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">  V2 - Reduced number of top frequencies <br> <br>
    Extracts list of top dominant frequencies: <br>
    Path do audio to analyse needs to be provided <br>
    Need to provide start and end time segment from which the dominant frequencies will be extracted. Segment time was decided after manual evaluation using <span style="color: red; font-weight: bold;">"listen_audio_file.py"</span>
</h3>

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from IPython.display import Audio, display


# ----------------- Configuration -----------------
# Provide the path to your audio file here
file_path = "./audio/washing_machine/becken_BWM5381IX/others/wm_becken_BWM5381IX_warm_delicate_30_0.wav"

# Define time segment to analyze (in seconds)
# You can modify these values to analyze different segments
start_time = 2703  # Start time in seconds
end_time =  2708  # Start time in seconds

# -------------------------------------------------

segment_duration = end_time - start_time

y, sr = librosa.load(file_path, sr=11025)

print(f"Audio loaded: {len(y)/sr:.2f} seconds total duration")
print(f"Sample rate: {sr} Hz")
print(f"\nAnalyzing segment from {start_time:.2f}s to {end_time:.2f}s ({segment_duration:.2f}s duration)")

start_sample = int(start_time * sr)
end_sample = int(end_time * sr)

if start_sample >= len(y):
    print(f"Error: Start time {start_time:.2f}s is beyond audio duration ({len(y)/sr:.2f}s)")
    exit()

if end_sample > len(y):
    print(f"Warning: End time {end_time:.2f}s exceeds audio duration, trimming to {len(y)/sr:.2f}s")
    end_sample = len(y)
    end_time = len(y) / sr

y_segment = y[start_sample:end_sample]

print(f"Extracted {len(y_segment)/sr:.2f} seconds of audio")

# Analysis parameters
frame_size = 3654
hop_size = 3664

# Step 1: Energy envelope
env = np.array([
    np.sum(np.abs(y_segment[i:i+frame_size])**2)
    for i in range(0, len(y_segment) - frame_size, hop_size)
])

if len(env) == 0:
    print("Error: Segment too short for analysis")
    exit()

env /= np.max(env)

# Step 2: Autocorrelation
auto = np.correlate(env, env, mode='full')[len(env):]
auto /= np.max(auto)

# Step 3: Estimate dominant repeating interval
peaks, _ = find_peaks(auto, height=0.3, distance=sr*1/hop_size)
if len(peaks) == 0:
    print("No repeating patterns found in this segment.")
    analysis_segment = y_segment
    print("Analyzing entire selected segment for frequency content...")
else:
    lag_frames = peaks[0]
    lag_seconds = lag_frames * hop_size / sr
    print(f"Estimated repeating cycle duration: {lag_seconds:.2f} seconds")

    cycle_samples = int(lag_seconds * sr)
    if cycle_samples * 2 > len(y_segment):
        print("Warning: Cycle too long for segment, using entire segment")
        analysis_segment = y_segment
    else:
        analysis_segment = y_segment[cycle_samples:2*cycle_samples]
        print(f"Analyzing first cycle ({lag_seconds:.2f}s duration)")

# FFT analysis
n_fft = 4096
fft = np.fft.rfft(analysis_segment, n=n_fft)
magnitudes = np.abs(fft)
magnitudes_db = 20 * np.log10(magnitudes + 1e-6)
freqs = np.fft.rfftfreq(n_fft, d=1/sr)

# Filter peaks in 5–40 Hz range
min_freq = 5
max_freq = 40
peaks, _ = find_peaks(magnitudes_db, height=np.mean(magnitudes_db))

if len(peaks) > 0:
    freqs_in_range = freqs[peaks]
    mags_in_range = magnitudes_db[peaks]
    mask = (freqs_in_range >= min_freq) & (freqs_in_range <= max_freq)
    filtered_freqs = freqs_in_range[mask]
    filtered_mags = mags_in_range[mask]
else:
    filtered_freqs = np.array([])
    filtered_mags = np.array([])

# Print results
print(f"\nDominant frequencies between {min_freq} Hz and {max_freq} Hz:")
if len(filtered_freqs) == 0:
    print("No significant frequencies found in this range.")
else:
    for i, (freq, mag) in enumerate(zip(filtered_freqs, filtered_mags), 1):
        print(f"{i}. {freq:.2f} Hz ({mag:.2f} dB)")

# Plotting
plt.figure(figsize=(15, 12))

# 1. Waveform
plt.subplot(3, 1, 1)
time_axis = np.linspace(start_time, start_time + len(y_segment)/sr, len(y_segment))
plt.plot(time_axis, y_segment)
plt.title(f'Waveform: {start_time:.2f}s to {end_time:.2f}s')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.grid(True)

# 2. Frequency spectrum
plt.subplot(3, 1, 2)
plt.plot(freqs, magnitudes_db, label='Frequency Spectrum', alpha=0.7)
plt.scatter(filtered_freqs, filtered_mags, color='red', s=50, zorder=5, label='5–40 Hz')

for i, (freq, mag) in enumerate(zip(filtered_freqs, filtered_mags), 1):
    plt.annotate(f'{freq:.1f} Hz', 
                 xy=(freq, mag), 
                 xytext=(10, 10), 
                 textcoords='offset points',
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
                 arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.title(f'Frequency Analysis: {start_time:.2f}s to {end_time:.2f}s')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.xlim(0, 100)  # Show full range of interest
plt.grid(True)
plt.legend()

# 3. Spectrogram
plt.subplot(3, 1, 3)
hop_length = 4096
n_fft = 8192
D = librosa.stft(y_segment, n_fft=n_fft, hop_length=hop_length)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
times = librosa.frames_to_time(np.arange(S_db.shape[1]), sr=sr, hop_length=hop_length) + start_time
freqs_spec = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

img = plt.imshow(S_db, aspect='auto', origin='lower', 
                 extent=[times[0], times[-1], freqs_spec[0], freqs_spec[-1]],
                 cmap='viridis')
plt.colorbar(img, format='%+2.0f dB')
plt.title(f'Spectrogram: {start_time:.2f}s to {end_time:.2f}s')
plt.xlabel('Time (seconds)')
plt.ylabel('Frequency (Hz)')
plt.ylim(0, 100)  # Focus on low frequencies
plt.tight_layout()
plt.show()

# Play audio
print(f"\nPlaying selected segment ({start_time:.2f}s to {end_time:.2f}s):")
display(Audio(y_segment, rate=sr))

# Summary
print(f"\n=== Analysis Summary ===")
print(f"Sample rate: {sr} Hz")
print(f"Time segment: {start_time:.2f}s - {end_time:.2f}s ({segment_duration:.2f}s)")
print(f"Sample range: {start_sample} - {end_sample}")
if len(filtered_freqs) > 0:
    print(f"Strongest frequency in 5–40 Hz: {filtered_freqs[0]:.2f} Hz")
print(f"Energy in segment: {np.sum(y_segment**2):.2e}")
print(f"Peak amplitude: {np.max(np.abs(y_segment)):.3f}")
